<a href="https://colab.research.google.com/github/Mriano29/hotel_demand_forecasting_system/blob/main/forecasting_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**`Notebook 3`: Sistema de Predicción de Cancelaciones Hoteleras**

## **1.0 — Construcción de la variable objetivo**

El objetivo de este modelo es predecir la ocupación futura del hotel (occupancy rate o número de habitaciones ocupadas) en una fecha determinada, utilizando información histórica y variables disponibles hasta el momento de la predicción.

Dado que el sistema está diseñado para simular un entorno real de Revenue Management en producción, se presta especial atención a la correcta construcción temporal del dataset, evitando cualquier forma de data leakage que incluya información posterior a la fecha objetivo.

El modelo se basa en una aproximación de series temporales supervisadas, donde cada observación representa un día de estancia y se utilizan variables agregadas de comportamiento de reservas, estacionalidad, demanda histórica y cancelaciones esperadas para generar predicciones realistas de la demanda futura del hotel.

### 1.1 — Carga de datos

In [115]:
import pandas as pd

url = "https://raw.githubusercontent.com/Mriano29/hotel_demand_forecasting_system/refs/heads/main/data/processed_data/processed_bookings.csv"

df = pd.read_csv(url)
original = df.copy()
df.head()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,...,assigned_room_type_G,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,7,0,1,0,0,0,0,0,75.0,...,False,False,False,False,False,False,False,False,True,False
1,0,20,0,2,0,0,0,0,0,62.0,...,False,False,False,False,False,False,False,False,False,True
2,0,20,0,2,0,0,0,7,0,62.0,...,False,False,False,False,False,False,False,False,True,False
3,1,20,0,2,0,0,0,0,0,62.0,...,False,False,False,False,False,False,False,False,False,True
4,0,20,0,2,0,0,0,0,0,62.0,...,False,False,False,False,False,False,False,False,False,True


### 1.2 — Carga del modelo de cancelación y configuración

Se carga un modelo XGBoost previamente entrenado para estimar la probabilidad de cancelación de cada reserva, junto con su configuración asociada (umbral de decisión).

In [116]:
import requests
import joblib
from io import BytesIO
import json

url_model = "https://raw.githubusercontent.com/Mriano29/hotel_demand_forecasting_system/main/ml/models/cancellations/xgboost_cancelation_model.pkl"
url_json = "https://raw.githubusercontent.com/Mriano29/hotel_demand_forecasting_system/refs/heads/main/ml/models/cancellations/cancelation_model_config.json"

# Cargar modelo
response = requests.get(url_model)
model_cancel = joblib.load(BytesIO(response.content))

# Cargar configuración
config = requests.get(url_json).json()

threshold = config["threshold"]

print("Modelo y configuración cargados correctamente")
print("Threshold:", threshold)

Modelo y configuración cargados correctamente
Threshold: 0.15


### 1.3 — Predicción de probabilidad de cancelación por reserva

En este paso aplicamos el modelo de cancelación previamente cargado sobre el dataset de reservas. El objetivo es enriquecer cada reserva con una probabilidad estimada de cancelación (`p_cancel`) y su complemento (`p_stay`), que representa la probabilidad de que la reserva finalmente se mantenga.

Primero extraemos las features exactas que el modelo espera, asegurando consistencia con el entrenamiento original. Luego construimos la matriz de entrada X_cancel usando únicamente esas columnas del dataset.

A partir de ahí, el modelo genera una probabilidad de cancelación por reserva, lo que nos permite transformar el problema de predicción binaria en una señal probabilística continua. Esta señal será clave posteriormente para estimar la demanda esperada en el nivel diario del forecasting.

In [117]:
# Features usadas por el modelo (extraídas desde XGBoost)
features = model_cancel.get_booster().feature_names

# Matriz de entrada
X_cancel = df[features]

# Probabilidad de cancelación
df["p_cancel"] = model_cancel.predict_proba(X_cancel)[:, 1]

# Probabilidad de que la reserva se mantenga
df["p_stay"] = 1 - df["p_cancel"]

df[["p_cancel", "p_stay"]].head()

,p_cancel,p_stay
0,0.018743,0.981257
1,0.319119,0.680881
2,0.346399,0.653601
3,0.319119,0.680881
4,0.319119,0.680881


### 1.4 — Expansión de reservas a nivel de estancia diaria

En este paso transformamos el dataset desde el nivel de reserva (booking-level) al nivel de estancia diaria (stay-day level), que es el formato necesario para construir una serie temporal de ocupación hotelera.

Cada reserva contiene una fecha de llegada (arrival_date) y una duración en noches (total_nights). A partir de esta información, generamos una secuencia de fechas consecutivas que representan cada noche en la que la reserva ocupa una habitación.

Esto se realiza creando una columna date mediante un rango de fechas diario desde la fecha de llegada durante el número de noches de la reserva. Posteriormente, utilizamos explode para convertir cada lista de fechas en filas individuales, de forma que cada fila represente una noche real de ocupación.

In [118]:
df["date"] = df.apply(
    lambda r: pd.date_range(
        r["arrival_date"],
        periods=int(r["total_nights"]),
        freq="D"
    ),
    axis=1
)

df_stays = df.explode("date")

### 1.5 — Cálculo del target de ocupación diaria esperada

En este paso construimos la variable objetivo del problema de forecasting: la ocupación diaria esperada del hotel.

Para ello, utilizamos el dataset expandido a nivel de estancia diaria (df_stays), donde cada fila representa una noche de estancia de una reserva. Como cada reserva tiene asociada una probabilidad de mantenerse (p_stay), esta se utiliza como peso para representar su contribución esperada a la ocupación real.

Posteriormente, agregamos todas las contribuciones por día utilizando groupby sobre la columna date, obteniendo así la ocupación total esperada para cada día del calendario. Este enfoque transforma el problema desde un nivel de reservas individuales a una serie temporal diaria adecuada para modelos de forecasting.

Finalmente, ordenamos la serie temporal para garantizar continuidad temporal en el dataset.

In [119]:
df_stays["weight"] = df_stays["p_stay"]

occupied_rooms = (
    df_stays
    .groupby("date")["weight"]
    .sum()
    .reset_index()
)

occupied_rooms.columns = ["date", "occupied_rooms"]

occupied_rooms = occupied_rooms.sort_values("date")

occupied_rooms.head()

,date,occupied_rooms
0,2015-07-01,82.382225
1,2015-07-02,115.014015
2,2015-07-03,95.875526
3,2015-07-04,121.256439
4,2015-07-05,119.958435


## **2.0 — Feature Engineering Temporal: Lags y Tendencias**

In [120]:
df_ocro = occupied_rooms.copy()

### 2.1 — Creación y merge con el dataset macro

En este paso construimos un dataset agregado de variables macroeconómicas a nivel diario, con el objetivo de integrarlas posteriormente como variables exógenas en el modelo de forecasting.

Dado que estas variables originalmente pueden estar a nivel de reserva o con múltiples registros por día, es necesario agregarlas correctamente al mismo nivel temporal que el target (date). Esto asegura consistencia temporal y evita desalineaciones entre features y la variable objetivo.

In [121]:
macro_cols = [
    "empleos_por_cada_100_plazas_alojativas",
    "ingresos_totales",
    "plazas",
    "tarifa_media_diaria",
    "tasa_de_ocupacion_por_habitacion",
    "tasa_de_ocupacion_por_plaza",
    "estancia_media",
    "pernoctaciones",
    "tmed",
    "prec"
]

macro_df = (
    original
    .groupby("arrival_date")[macro_cols]
    .mean()
    .reset_index()
    .rename(columns={"arrival_date": "date"})
)

In [122]:
#Aseguramos que esten en formato fecha
df_ocro["date"] = pd.to_datetime(df_ocro["date"])
macro_df["date"] = pd.to_datetime(macro_df["date"])

#Hacemos el merge
df_ocro = df_ocro.merge(macro_df, on="date", how="left")
df_ocro.head()

,date,occupied_rooms,empleos_por_cada_100_plazas_alojativas,ingresos_totales,plazas,tarifa_media_diaria,tasa_de_ocupacion_por_habitacion,tasa_de_ocupacion_por_plaza,estancia_media,pernoctaciones,tmed,prec
0,2015-07-01,82.382225,13.37,213641489.9,412368.0,62.74,67.73,58.9,4.589888,828126.0,23.1,0.0
1,2015-07-02,115.014015,13.37,213641489.9,412368.0,62.74,67.73,58.9,4.589888,828126.0,23.1,0.0
2,2015-07-03,95.875526,13.37,213641489.9,412368.0,62.74,67.73,58.9,4.589888,828126.0,23.1,0.0
3,2015-07-04,121.256439,13.37,213641489.9,412368.0,62.74,67.73,58.9,4.589888,828126.0,23.1,0.0
4,2015-07-05,119.958435,13.37,213641489.9,412368.0,62.74,67.73,58.9,4.589888,828126.0,23.1,0.0


### 2.2 — Creación de variables de memoria temporal (lags)

En este paso construimos variables de retardo (lags) a partir de la serie temporal de ocupación diaria del hotel. Estas variables permiten al modelo capturar dependencias temporales y patrones históricos de demanda, que son fundamentales en problemas de forecasting.

Cada lag representa el valor de ocupación en un punto específico del pasado:

- `lag_1`: ocupación del día anterior
- `lag_7`: ocupación de hace una semana (captura estacionalidad semanal)
- `lag_14`: patrón de corto-medio plazo
- `lag_30`: tendencia mensual

In [123]:
df_ocro["lag_1"] = df_ocro["occupied_rooms"].shift(1)
df_ocro["lag_7"] = df_ocro["occupied_rooms"].shift(7)
df_ocro["lag_14"] = df_ocro["occupied_rooms"].shift(14)
df_ocro["lag_30"] = df_ocro["occupied_rooms"].shift(30)

### 2.3 — Creación de variables de tendencia (rolling features)

En este paso añadimos variables de medias móviles para capturar tendencias suavizadas en la ocupación hotelera. Estas features ayudan al modelo a entender el comportamiento reciente de la demanda eliminando ruido diario y destacando patrones más estables.

Para evitar leakage temporal, aplicamos un desplazamiento (`shift(1)`) antes de calcular las medias móviles, asegurando que cada valor solo utilice información del pasado y nunca del día actual.

Las ventanas utilizadas representan diferentes horizontes de tendencia:

- `roll_mean_7`: tendencia semanal reciente
- `roll_mean_14`: tendencia quincenal
- `roll_mean_30`: tendencia mensual

In [124]:
df_ocro["roll_mean_7"] = df_ocro["occupied_rooms"].shift(1).rolling(7).mean()
df_ocro["roll_mean_14"] = df_ocro["occupied_rooms"].shift(1).rolling(14).mean()
df_ocro["roll_mean_30"] = df_ocro["occupied_rooms"].shift(1).rolling(30).mean()

### 2.4 — Eliminación de valores nulos tras la creación de features temporales

En este paso limpiamos el dataset eliminando las filas que contienen valores nulos (NaN) generados durante la creación de variables temporales como lags y medias móviles.

In [125]:
df_ocro = df_ocro.dropna().copy()

## **3.0 — Preparación de datasets de entrenamiento y validación para forecasting temporal**

### 3.1 — División temporal del dataset en entrenamiento y test

Se define una fecha de corte (cut_date) que marca el punto a partir del cual los datos pasan a formar parte del conjunto de test. Todo lo anterior a esa fecha se utiliza para entrenar el modelo, mientras que los datos posteriores se reservan para evaluar su capacidad de predicción en el futuro.

In [126]:
cut_date = "2016-01-01"

train_df = df_ocro[df_ocro["date"] < cut_date]
test_df  = df_ocro[df_ocro["date"] >= cut_date]

### 3.2 — Separación de variables predictoras y variable objetivo

In [127]:
target = "occupied_rooms"

X_train = train_df.drop(columns=[target, "date"])
y_train = train_df[target]

X_test = test_df.drop(columns=[target, "date"])
y_test = test_df[target]

## **4.0 — Entrenamiento y evaluación del modelo**

### 4.1 — Entrenamiento baseline con prophet


En este paso se entrena un modelo Prophet como referencia inicial para el problema de forecasting de ocupación diaria. Este modelo sirve como baseline para comparar posteriormente el rendimiento de modelos más complejos como XGBoost.

Prophet requiere un formato específico de entrada (ds, y) y se entrena únicamente con el conjunto de entrenamiento para respetar la estructura temporal del problema.

In [128]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# =========================
# Preparación de datos
# =========================
prophet_train = train_df[["date", "occupied_rooms"]].rename(
    columns={"date": "ds", "occupied_rooms": "y"}
)

prophet_test = test_df[["date", "occupied_rooms"]].rename(
    columns={"date": "ds", "occupied_rooms": "y"}
)

# =========================
# Modelo Prophet
# =========================
model = Prophet()
model.fit(prophet_train)

# =========================
# Predicción
# =========================
future = model.make_future_dataframe(periods=len(prophet_test), freq="D")
forecast = model.predict(future)

# Filtrar solo test
pred_test = forecast.merge(prophet_test, on="ds", how="inner")

# =========================
# Métricas
# =========================
mae = mean_absolute_error(pred_test["y"], pred_test["yhat"])
rmse = np.sqrt(mean_squared_error(pred_test["y"], pred_test["yhat"]))

print("MAE:", mae)
print("RMSE:", rmse)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


MAE: 741.0216958414377
RMSE: 822.211005366916


Los resultados obtenidos muestran un error elevado tanto en MAE como en RMSE, lo que indica que el modelo no está siendo capaz de ajustar adecuadamente la dinámica de la serie de ocupación diaria.

Este comportamiento es esperable en problemas como el nuestro, donde la serie no depende únicamente de estacionalidad global, sino también de factores más complejos como:

- Comportamiento de reservas y cancelaciones
- Efectos de corto plazo (lags y autocorrelación fuerte)
- Variables exógenas (macro y turismo)
- Cambios bruscos en la demanda

Prophet, al ser un modelo univariante en esta configuración, no puede capturar este tipo de relaciones, lo que limita su capacidad predictiva en este caso.

### 4.2 — Entrenamiento con XGBoost


En este paso se entrena un modelo XGBoost Regressor para el problema de forecasting de ocupación hotelera. A diferencia de Prophet, este enfoque utiliza un conjunto de variables explicativas derivadas del pipeline híbrido, lo que permite capturar relaciones más complejas en los datos.

El modelo se entrena utilizando únicamente información disponible en el pasado, eliminando la variable objetivo (occupied_rooms) y la fecha (date) del conjunto de entrada. Esto garantiza una correcta separación entre features y target.

El objetivo es evaluar cuánto mejora el rendimiento al incorporar información estructurada como efectos temporales y comportamiento de la demanda.

In [129]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

target = "occupied_rooms"

# Separación X / y
X_train = train_df.drop(columns=[target, "date"])
y_train = train_df[target]

X_test = test_df.drop(columns=[target, "date"])
y_test = test_df[target]

# Modelo base XGBoost
model_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entrenamiento
model_xgb.fit(X_train, y_train)

# Predicción
y_pred = model_xgb.predict(X_test)

# Métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 88.88776397705078
RMSE: 102.84126212426848


El modelo XGBoost mejora de forma significativa el rendimiento respecto a Prophet, lo que indica que la inclusión de variables derivadas del comportamiento de reservas (lags, rolling, cancelaciones y variables macroeconómicas) aporta información relevante para explicar la ocupación diaria.

Esto confirma que el problema no es puramente una serie temporal univariante, sino un sistema multivariable donde la dinámica histórica y el contexto externo juegan un papel clave en la predicción.

### 4.3 — Entrenamiento con LGBM

En este paso se entrena un modelo LightGBM Regressor para el problema de forecasting de ocupación hotelera, utilizando el mismo conjunto de variables explicativas que en el caso de XGBoost (lags, rolling, variables macroeconómicas y probabilidad de cancelación).

LightGBM es un modelo basado en gradient boosting que suele ofrecer un rendimiento competitivo en problemas tabulares, especialmente cuando existen interacciones no lineales y variables correlacionadas.

El objetivo aquí es comparar directamente su capacidad predictiva frente a XGBoost bajo el mismo esquema de validación temporal.

In [130]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

target = "occupied_rooms"

X_train = train_df.drop(columns=[target, "date"])
y_train = train_df[target]

X_test = test_df.drop(columns=[target, "date"])
y_test = test_df[target]

model_lgb = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_lgb.fit(X_train, y_train)

y_pred = model_lgb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 428
[LightGBM] [Info] Number of data points in the train set: 154, number of used features: 17
[LightGBM] [Info] Start training from score 260.000970
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

LightGBM mejora aún más el rendimiento respecto a XGBoost, lo que sugiere que este modelo es más eficiente capturando las interacciones entre variables temporales, macroeconómicas y de comportamiento de reservas.

Esto refuerza la hipótesis de que el problema de forecasting se beneficia claramente de modelos basados en árboles con features estructuradas, frente a modelos puramente de series temporales como Prophet.

### 4.4 — Validación temporal comparativa: XGBoost vs LightGBM (walk-forward)

En este paso se evalúan ambos modelos bajo múltiples cortes temporales progresivos. El objetivo es comprobar la estabilidad del rendimiento en distintos periodos históricos, evitando depender de un único split train/test.

In [132]:
from sklearn.metrics import mean_absolute_error
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

cut_dates = [
    "2016-01-01",
    "2016-07-01",
    "2017-01-01"
]

maes_xgb = []
maes_lgb = []

for cut in cut_dates:
    train = df_ocro[df_ocro["date"] < cut]
    test = df_ocro[df_ocro["date"] >= cut]

    X_train = train.drop(columns=["occupied_rooms", "date"])
    y_train = train["occupied_rooms"]

    X_test = test.drop(columns=["occupied_rooms", "date"])
    y_test = test["occupied_rooms"]

    # =====================
    # XGBoost
    # =====================
    model_xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model_xgb.fit(X_train, y_train)
    preds_xgb = model_xgb.predict(X_test)
    maes_xgb.append(mean_absolute_error(y_test, preds_xgb))

    # =====================
    # LightGBM
    # =====================
    model_lgb = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.05,
        random_state=42
    )

    model_lgb.fit(X_train, y_train)
    preds_lgb = model_lgb.predict(X_test)
    maes_lgb.append(mean_absolute_error(y_test, preds_lgb))

    print(f"\nCut {cut}")
    print(f"XGBoost MAE: {maes_xgb[-1]}")
    print(f"LightGBM MAE: {maes_lgb[-1]}")

print("\n====================")
print("RESULTADOS FINALES")
print("====================")

print("XGBoost MAE medio:", np.mean(maes_xgb))
print("XGBoost STD:", np.std(maes_xgb))

print("LightGBM MAE medio:", np.mean(maes_lgb))
print("LightGBM STD:", np.std(maes_lgb))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000028 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 428
[LightGBM] [Info] Number of data points in the train set: 154, number of used features: 17
[LightGBM] [Info] Start training from score 260.000970
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

Los resultados muestran diferencias claras entre ambos enfoques:

- XGBoost presenta un error medio más alto y una mayor variabilidad entre cortes temporales.
- LightGBM consigue un error medio menor y una desviación estándar más baja, lo que indica un comportamiento más estable y consistente en el tiempo.

Estos resultados sugieren que, para este problema de forecasting de ocupación hotelera, el modelo LightGBM ofrece un mejor equilibrio entre precisión y estabilidad temporal.

La menor variabilidad del error indica que el modelo generaliza mejor a distintos periodos históricos, lo cual es especialmente importante en problemas de demanda donde existen cambios estacionales y dinámicas no estacionarias.

### 4.5 — Optimización de hiperparámetros con validación temporal (LightGBM)

En esta fase se realiza una búsqueda aleatoria de hiperparámetros para el modelo LightGBM, utilizando una estrategia de validación tipo walk-forward con múltiples cortes temporales.

El objetivo no es optimizar el modelo en un único split, sino evaluar cada combinación de hiperparámetros en distintos periodos históricos, simulando un entorno más realista de predicción en producción.

Para cada configuración se calcula:

- MAE medio en los distintos cortes temporales (precisión global)
- Desviación estándar del MAE (estabilidad temporal)
- Score combinado (MAE + penalización por inestabilidad)

Este enfoque permite priorizar modelos no solo precisos, sino también robustos a cambios en el tiempo.

In [139]:
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
import random
import warnings

# =========================
# Silenciar warnings
# =========================
warnings.filterwarnings("ignore")

cut_dates = [
    "2016-01-01",
    "2016-07-01",
    "2017-01-01"
]

param_space = {
    "n_estimators": [300, 500, 800, 1200],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 10, 15],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_samples": [10, 20, 50]
}

def sample_params():
    return {k: random.choice(v) for k, v in param_space.items()}

results = []

for i in range(20):

    params = sample_params()
    maes = []

    for cut in cut_dates:

        train = df_ocro[df_ocro["date"] < cut]
        test = df_ocro[df_ocro["date"] >= cut]

        X_train = train.drop(columns=["occupied_rooms", "date"])
        y_train = train["occupied_rooms"]

        X_test = test.drop(columns=["occupied_rooms", "date"])
        y_test = test["occupied_rooms"]

        model = LGBMRegressor(**params, random_state=42, verbosity=-1)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        maes.append(mean_absolute_error(y_test, preds))

    results.append({
        "mae_mean": np.mean(maes),
        "mae_std": np.std(maes),
        "score": np.mean(maes) + np.std(maes),
        **params
    })

# =========================
# Tabla final ordenada
# =========================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("score").reset_index(drop=True)

results_df.head()

,mae_mean,mae_std,score,n_estimators,learning_rate,num_leaves,max_depth,subsample,colsample_bytree,min_child_samples
0,41.855685,15.381305,57.236990,500,0.01,15,15,0.8,1.0,20
1,44.580421,13.067074,57.647495,500,0.10,15,15,0.8,1.0,20
2,41.345664,16.513596,57.859260,800,0.01,15,10,0.6,0.8,10
3,42.711103,15.209515,57.920618,300,0.03,31,15,0.6,0.8,10
4,43.127417,14.953323,58.080740,300,0.03,63,15,1.0,1.0,10


## **5.0 — Entrenamiento final del modelo (LightGBM optimizado)**

En esta fase se entrena el modelo final de LightGBM utilizando los mejores hiperparámetros obtenidos mediante validación temporal y búsqueda aleatoria.

Este modelo ya no se evalúa sobre múltiples cortes, sino que se ajusta con la configuración óptima seleccionada para ser utilizado como modelo definitivo del sistema de forecasting.

In [140]:
from lightgbm import LGBMRegressor

# =========================
# Mejores hiperparámetros
# =========================
best_params = {
    "n_estimators": 500,
    "learning_rate": 0.01,
    "num_leaves": 15,
    "max_depth": 15,
    "subsample": 0.8,
    "colsample_bytree": 1.0,
    "min_child_samples": 20
}

# =========================
# Separación X / y final
# =========================
X = df_ocro.drop(columns=["occupied_rooms", "date"])
y = df_ocro["occupied_rooms"]

# =========================
# Entrenamiento final
# =========================
final_model = LGBMRegressor(**best_params, random_state=42, verbosity=-1)
final_model.fit(X, y)

LGBMRegressor(learning_rate=0.01, max_depth=15, n_estimators=500, num_leaves=15,
              random_state=42, subsample=0.8, verbosity=-1)

### 5.1 — Exportado del modelo final

In [141]:
joblib.dump(final_model, "occupancy_model.pkl")

['occupancy_model.pkl']